In [12]:
import requests
import pandas as pd
import time
import re

# --- API SETTINGS ---
APP_ID = '7b5447e8'
APP_KEY = '8043d3c5107de92d7303ef1b512320b2'
COUNTRY = 'us' 

# --- HELPER FUNCTIONS FOR PARSING DESCRIPTION ---
def extract_skills(description):
    """
    Scans the description for a broad range of skills, including Healthcare.
    """
    skills_database = [
        # IT and Tech
        'python', 'sql', 'java', 'javascript', 'html', 'css', 'c++', 'aws', 'azure', 
        'machine learning', 'data analysis', 'tableau', 'power bi', 'jira',
        
        # Business and Management
        'excel', 'agile', 'scrum', 'marketing', 'sales', 'seo', 'leadership', 
        'communication', 'project management', 'crm', 'salesforce', 'customer service',
        'troubleshooting', 'problem solving', 'teamwork', 'planning', 'reporting',
        'operations', 'logistics', 'presentation', 'training', 'research', 'management',
        
        # Healthcare and Medicine (NEW)
        'patient care', 'cpr', 'bls', 'nursing', 'emr', 'ehr', 'medical terminology', 
        'triage', 'medication administration', 'rehabilitation', 'clinical', 'hipaa'
    ]
    
    found_skills = []
    desc_lower = str(description).lower()
    
    for skill in skills_database:
        if re.search(r'\b' + re.escape(skill) + r'\b', desc_lower):
            found_skills.append(skill.title())
            
    return ", ".join(found_skills)

def extract_education(description):
    """
    Uses flexible regex to catch various formats of education requirements.
    """
    education_patterns = {
        'High School / GED': r'\b(high school|ged)\b',
        'Associate': r'\b(associate\'?s?|a\.a\.|a\.s\.)\b',
        'Bachelor': r'\b(bachelor\'?s?|b\.a\.|b\.s\.|bsc|b\.tech|degree|bsn)\b', # Added BSN for Nursing
        'Master': r'\b(master\'?s?|m\.a\.|m\.s\.|mba|msn)\b', # Added MSN
        'PhD / MD': r'\b(ph\.?d|doctorate|m\.d\.|d\.o\.)\b' # Added MD/DO for Medicine
    }
    
    found_edu = []
    desc_lower = str(description).lower()
    
    for label, pattern in education_patterns.items():
        if re.search(pattern, desc_lower):
            found_edu.append(label)
            
    return ", ".join(found_edu)

# --- MAIN PARSER FUNCTION ---
def fetch_adzuna_jobs_by_query(queries, pages_per_query=10, results_per_page=50):
    """
    Fetches a balanced dataset by cycling through different job domains.
    """
    jobs_data = []
    total_expected = len(queries) * pages_per_query * results_per_page
    print(f"Starting targeted job download ({total_expected} expected)...")
    
    for query in queries:
        print(f"\nFetching jobs for: '{query}'...")
        for page in range(1, pages_per_query + 1):
            url = f"https://api.adzuna.com/v1/api/jobs/{COUNTRY}/search/{page}"
            
            params = {
                'app_id': APP_ID,
                'app_key': APP_KEY,
                'results_per_page': results_per_page,
                'what': query,
                'content-type': 'application/json'
            }
            
            response = requests.get(url, params=params)
            
            if response.status_code == 200:
                data = response.json()
                results = data.get('results', [])
                
                for job in results:
                    description_text = job.get('description', '')
                    
                    jobs_data.append({
                        'job_title': job.get('title', ''),
                        'company_name': job.get('company', {}).get('display_name', ''),
                        'category': job.get('category', {}).get('label', ''),
                        'description': description_text,
                        'location': ', '.join(job.get('location', {}).get('area', [])),
                        'salary_min': job.get('salary_min', None),
                        'salary_max': job.get('salary_max', None),
                        'contract_time': job.get('contract_time', ''),
                        'contract_type': job.get('contract_type', ''),
                        'created': job.get('created', ''),
                        'job_url': job.get('redirect_url', ''),
                        
                        'required_skills': extract_skills(description_text),
                        'required_degree': extract_education(description_text)
                    })
                
                print(f"'{query}' - Page {page} processed. Total collected: {len(jobs_data)}")
            else:
                print(f"Error on page {page} for '{query}': HTTP {response.status_code}")
            
            time.sleep(2)
            
    return pd.DataFrame(jobs_data)

In [13]:
# --- MAIN EXECUTION BLOCK ---

# Now 21 queries including "Healthcare"
search_queries = [
    "Software Engineer", "Data Analyst", "Marketing", "Sales Manager",
    "Accountant", "Human Resources", "Project Manager", "Customer Service",
    "Mechanical Engineer", "Operations Manager", "Graphic Designer",
    "Financial Analyst", "Teacher", "Administrative Assistant",
    "Business Analyst", "Consultant", "Supply Chain", "Retail",
    "Electrician", "Legal", "Healthcare"
]

# 21 queries * 10 pages * 50 results = 10,500 jobs expected
final_jobs_df = fetch_adzuna_jobs_by_query(
    queries=search_queries, 
    pages_per_query=10, 
    results_per_page=50
)

# Clean up possible duplicates
final_jobs_df.drop_duplicates(subset=['job_url'], inplace=True)

print(f"\nCollection complete. Total unique jobs collected: {len(final_jobs_df)}")

# Save to a CSV file
file_name = 'adzuna_diverse_jobs_with_medical.csv'
final_jobs_df.to_csv(file_name, index=False, encoding='utf-8')
print(f"Data successfully saved to file: '{file_name}'")

# Check results for Healthcare specifically
final_jobs_df[final_jobs_df['category'].str.contains('Healthcare', na=False, case=False)][['job_title', 'required_skills', 'required_degree']].head(10)

Starting targeted job download (10500 expected)...

Fetching jobs for: 'Software Engineer'...
'Software Engineer' - Page 1 processed. Total collected: 50
'Software Engineer' - Page 2 processed. Total collected: 100
'Software Engineer' - Page 3 processed. Total collected: 150
'Software Engineer' - Page 4 processed. Total collected: 200
'Software Engineer' - Page 5 processed. Total collected: 250
'Software Engineer' - Page 6 processed. Total collected: 300
'Software Engineer' - Page 7 processed. Total collected: 350
'Software Engineer' - Page 8 processed. Total collected: 400
'Software Engineer' - Page 9 processed. Total collected: 450
'Software Engineer' - Page 10 processed. Total collected: 500

Fetching jobs for: 'Data Analyst'...
'Data Analyst' - Page 1 processed. Total collected: 550
'Data Analyst' - Page 2 processed. Total collected: 600
'Data Analyst' - Page 3 processed. Total collected: 650
'Data Analyst' - Page 4 processed. Total collected: 700
'Data Analyst' - Page 5 processed.

,job_title,required_skills,required_degree
817,Senior Data Analyst,"Reporting, Clinical",
1445,Marketing Director,Marketing,
1458,Practice Relations/Marketing Ambassador,"Marketing, Planning",
1503,"Area Sales Manager, Nephrology Houston",,
1538,"Area Sales Manager, Nephrology Washington, DC",,
2894,Human Resources Manager,,
2895,Human Resources (HR) Coordinator,,
4518,Operations Manager,Operations,
4526,Operations Manager - Housekeeping Services,"Leadership, Operations, Management",
4542,Practice Operations Manager,Operations,
